# Testar um PCAP gerado pelo simulador

Carrega um **PCAP gerado** pelo `someip-traffic-simulator` (+ o `.labels.npy` de ground truth),
**extrai** as features `content_ext`, **treina/testa** um XGBoost multiclasse (70/30) e mostra
**métricas + matriz de confusão**. Serve para ver o comportamento do IDS no SEU tráfego.

**Você só precisa gerar o PCAP** (no simulador: `python run_scenario.py scenarios/...yaml`) e
apontar o caminho aqui (ou fazer upload no Colab). O simulador salva `<pcap>.labels.npy` junto.

In [ ]:
# 0) Setup — clona o repo (Colab) ou assume a raiz do repo (local)
import os
REPO = 'someip-ids-multiclass-contentext'
if not os.path.exists('eval_pcap.py'):
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    elif os.path.isdir(REPO):
        os.chdir(REPO)
    else:
        os.system(f'git clone -q https://github.com/GuilhermeFrick/{REPO}.git')
        os.chdir(REPO)
!pip -q install scapy xgboost scikit-learn matplotlib numpy
print('cwd:', os.getcwd())

In [ ]:
# 1) Aponte para o SEU PCAP gerado (+ labels). No Colab, faça upload e ajuste o caminho.
PCAP   = 'traces/dataset.pcap'          # <-- ajuste aqui
LABELS = PCAP + '.labels.npy'           # padrão do simulador

# (Colab) descomente para subir os arquivos pelo navegador:
# from google.colab import files
# up = files.upload()              # selecione o .pcap e o .pcap.labels.npy
# PCAP = [f for f in up if f.endswith('.pcap')][0]; LABELS = PCAP + '.labels.npy'

assert os.path.exists(PCAP), f'PCAP não encontrado: {PCAP}'
assert os.path.exists(LABELS), f'labels não encontrado: {LABELS}'
print('PCAP  :', PCAP)
print('labels:', LABELS)

In [ ]:
# 2) Extrai content_ext (byte-models ajustados no NORMAL do próprio PCAP)
import numpy as np
from eval_pcap import load_pcap, ORDER

X, ymap, _models = load_pcap(PCAP, LABELS, None)
u, c = np.unique(ymap, return_counts=True)
print('X:', X.shape, '(16 features content_ext)')
print('rótulos:', dict(zip(u.tolist(), c.tolist())))

In [ ]:
# 3) Treina/testa 70/30 e mostra as MÉTRICAS
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from xgboost import XGBClassifier

classes = [k for k in ORDER + ['attack'] if (ymap == k).sum() >= 6]
cid = {c: i for i, c in enumerate(classes)}
keep = np.isin(ymap, classes)
Xk, yk = X[keep], np.array([cid[c] for c in ymap[keep]])
xmin, xmax = Xk.min(0), Xk.max(0)
Xn = ((Xk - xmin) / np.where(xmax > xmin, xmax - xmin, 1.0)).astype(np.float32)
Xtr, Xte, ytr, yte = train_test_split(Xn, yk, test_size=0.30, random_state=0, stratify=yk)

clf = XGBClassifier(objective='multi:softprob', num_class=len(classes), n_estimators=300,
                    max_depth=6, learning_rate=0.3, tree_method='hist', n_jobs=-1,
                    eval_metric='mlogloss')
clf.fit(Xtr, ytr)
y_pred = clf.predict_proba(Xte).argmax(1)   # multi:softprob -> predict_proba + argmax
print(classification_report(yte, y_pred, labels=range(len(classes)),
                            target_names=classes, digits=4, zero_division=0))
print('macro-F1:', round(f1_score(yte, y_pred, average='macro'), 4))

In [ ]:
# 4) MATRIZ DE CONFUSÃO (inline)
import matplotlib.pyplot as plt
cm = confusion_matrix(yte, y_pred, labels=range(len(classes)))
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
N = len(classes)
fig, ax = plt.subplots(figsize=(1.2*N+3, 1.0*N+3))
im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(N)); ax.set_yticks(range(N))
ax.set_xticklabels(classes, rotation=45, ha='right'); ax.set_yticklabels(classes)
ax.set_xlabel('Predito'); ax.set_ylabel('Verdadeiro')
ax.set_title('IDS sobre o PCAP gerado — matriz de confusão')
for a in range(N):
    for b in range(N):
        ax.text(b, a, f'{cmn[a,b]*100:.0f}%', ha='center', va='center', fontsize=8,
                color='white' if cmn[a,b] > 0.5 else 'black')
fig.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout(); plt.show()

## Leitura

- macro-F1 **alto** → o tráfego gerado é **coerente e aprendível** (os ataques têm assinatura
  comportamental distinta do normal). É o teste de fechamento de ciclo do gerador.
- Veja na matriz **quais classes se confundem** — costuma ser o `mitm`/relay com o `normal`
  (mesmo padrão do dataset do Kim, pois o relay republica tráfego legítimo).

### Próximo passo — zero-day
Gere dois PCAPs (treino com ataques conhecidos, teste com um ataque **novo**) e rode via script:
```bash
python eval_pcap.py traces/train_known.pcap --test-pcap traces/test_novel.pcap --binary
```
Ele treina nos conhecidos e mede a **detecção do tipo inédito** (normal vs ataque).